In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../temp/demotermine_cleaned_2022_06_20_DE.csv', parse_dates=['date', 'edit_date'])

# only german

In [3]:
df = df[df['lang'] == 'de']

# min, max text length

In [4]:
df.text_length.describe()

count    22803.000000
mean       280.676008
std        422.848128
min          4.000000
25%         75.000000
50%        134.000000
75%        317.000000
max       4155.000000
Name: text_length, dtype: float64

In [5]:
df = df[(df['text_length'] >= 100)]# & (df['text_length'] <= 400)]

In [6]:
len(df)

14605

In [7]:
df.text_length.sum() // 500000 # DeeplAI translates 500.000 chars for free per month

11

## Identify similar messages via Levenshtein Distance

In [8]:
from itertools import combinations
import Levenshtein as lev

In [9]:
len(df)

14605

In [10]:
lev_df = pd.DataFrame() 
new_df = pd.DataFrame(combinations(df['cleaned_text'],2), columns=["text_1", "text_2"]) 
new_df["lev_score"] = new_df.apply(lambda x: lev.ratio(x[0],x[1]), axis=1) 
lev_df = pd.concat([lev_df, new_df], axis=0).reset_index(drop=True)

In [11]:
# Get message ids
lev_df_ids = pd.DataFrame() 
new_df_ids = pd.DataFrame(combinations(df['message_id'],2), columns=["message_id_1", "message_id_2"]) 
lev_df_ids = pd.concat([lev_df_ids, new_df_ids], axis=0).reset_index(drop=True)

In [12]:
# combine lev results with message ids 
levs = pd.DataFrame()
levs = pd.concat([lev_df,lev_df_ids], axis=1)

In [13]:
levs.head()

,text_1,text_2,lev_score,message_id_1,message_id_2
0,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERBOT...,0.345029,15971,6885
1,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,Stuttgart-Stream von Stephan Bergmann u.a. mit...,0.293515,15971,4836
2,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,Demo-Termine von Zum Redaktionsschluss um 00:0...,0.256410,15971,3105
3,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,"FREITAG ! ! Es ist nicht genug zu wollen , man...",0.253548,15971,10783
4,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,Bitte teilen Demo in Halle ( Saale ): Für Sonn...,0.277895,15971,9261


In [14]:
# get ids of messages over certain threshold
second_ids = levs[levs['lev_score'] >= 0.8].message_id_2.tolist()

In [15]:
len(set(second_ids))

3484

In [16]:
ids_to_drop = list(set(second_ids))

In [17]:
len(ids_to_drop)

3484

In [18]:
result_df = df[~df.message_id.isin(ids_to_drop)]

In [19]:
len(result_df)

11121

In [20]:
result_df.to_csv('../temp/data_selection_2022_06_20_after_levenshtein.csv')

In [21]:
result_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11121 entries, 0 to 22802
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Unnamed: 0.2         11121 non-null  int64         
 1   Unnamed: 0.1         11121 non-null  int64         
 2   Unnamed: 0           11121 non-null  int64         
 3   channel_name         11121 non-null  object        
 4   channel_id           11121 non-null  int64         
 5   channel_description  11121 non-null  object        
 6   message_id           11121 non-null  int64         
 7   from_id              0 non-null      float64       
 8   via_bot_id           2 non-null      float64       
 9   date                 11121 non-null  datetime64[ns]
 10  edit_date            10378 non-null  datetime64[ns]
 11  text                 11121 non-null  object        
 12  forwards             11121 non-null  float64       
 13  fwd_from             963 non-nu

In [22]:
columns_to_keep = ['id', 'message_id', 'date', 'cleaned_text', 'lang', 'text_length']

In [23]:
result_df = result_df[columns_to_keep]

In [24]:
result_df.head(20)

,id,message_id,date,cleaned_text,lang,text_length
0,Demotermine1250288610159712021-08-29 11:10:12,15971,2021-08-29 11:11:25,Berlin 29.08.2021 um 13 : 10 Uhr In ungefähr 2...,de,192
1,Demotermine125028861068852021-02-27 07:11:32,6885,2021-02-27 07:11:14,27 Feb : Sternmarsch – Graz BEHÖRDLICH VERBOT...,de,150
3,Demotermine125028861048362021-01-01 03:57:43,4836,2021-01-01 03:50:02,Stuttgart-Stream von Stephan Bergmann u.a. mit...,de,101
4,Demotermine125028861031052020-10-09 13:28:27,3105,2020-10-09 13:28:26,Demo-Termine von Zum Redaktionsschluss um 00:0...,de,744
5,Demotermine1250288610107832021-05-21 01:25:13,10783,2021-05-21 01:23:05,"FREITAG ! ! Es ist nicht genug zu wollen , man...",de,865
6,Demotermine125028861092612021-04-19 04:53:00,9261,2021-04-19 04:03:37,Bitte teilen Demo in Halle ( Saale ): Für Sonn...,de,758
7,Demotermine125028861091852021-04-15 22:59:42,9185,2021-04-15 22:59:04,"Liebe Freude der Grundrechte , wir haben einen...",de,498
8,Demotermine12502886109284,9284,2021-04-19 18:01:21,"LETZTER Halt der "" Legendär-Tour "" : ! ! BERLI...",de,692
9,Demotermine1250288610111052021-05-27 07:00:29,11105,2021-05-27 07:00:28,Demo # MobilitätswendeJetzt – Lobauautobahn & ...,de,106
11,Demotermine1250288610175472021-09-09 16:32:09,17547,2021-09-09 16:32:54,Das kommende Desaster : Rot-Rot-Grün mit Schol...,de,125


In [25]:
result_df.text_length.sum()

4905753

In [26]:
len(result_df)

11121

In [27]:
result_df.to_csv('../temp/data_selection_final_2022_06_20.csv')

In [28]:
result_df = pd.read_csv('../temp/data_selection_final_2022_06_20.csv')

# select sample 

In [29]:
twenty_percent = int(len(result_df) / 100 * 20)
sample_df = result_df.sample(twenty_percent, random_state=42)
sample_df.to_csv('../temp/sample_data_2022_06_20.csv')
len(sample_df)
#sample_df.text_length.sum()

2224

In [94]:
# texts = result_df.cleaned_text.values

In [58]:
#for t in texts: 
 #   print(t, '\n\n')